# HMS Harmful Brain Activity Classification — Inference
**Model**: EfficientNetV2-S + nnAudio trainable STFT + GRU (v9)  
**Input**: 16-channel bipolar EEG → multi-scale spectrogram (wide 50s + zoom 10s)  
**Preprocessing**: Bandpass 0.5–20 Hz → per-channel normalization

Upload your trained model checkpoints as a Kaggle dataset.


In [1]:
!pip install /kaggle/input/datasets/parapapapam/nnaudio031/*.whl -q --no-deps

import os
import math
import random
import gc
from pathlib import Path
from glob import glob

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast

import timm
from nnAudio.features import STFT
from scipy.signal import butter, sosfiltfilt

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


PyTorch: 2.9.0+cu126
CUDA available: True
GPU: Tesla P100-PCIE-16GB


## Configuration

In [2]:
class CFG:
    seed = 42
    device = "cuda" if torch.cuda.is_available() else "cpu"
    num_classes = 6
    class_names = ["Seizure", "LPD", "GPD", "LRDA", "GRDA", "Other"]

    # EEG
    eeg_sample_rate = 200
    eeg_duration = 50
    eeg_samples = eeg_sample_rate * eeg_duration  # 10000
    num_bipolar_channels = 16

    # Bandpass
    bandpass_low = 0.5
    bandpass_high = 20.0
    bandpass_order = 4

    # STFT
    n_fft = 128
    hop_length = 128
    trainable_stft = True
    zoom_duration = 10
    zoom_hop_length = 32
    freq_crop_hz = 20.0

    # Model
    backbone = "tf_efficientnetv2_s"
    gru_hidden = 128
    gru_layers = 1
    dropout = 0.35

    # Inference
    batch_size = 16
    num_workers = 2
    use_amp = False

print(f"Device: {CFG.device}")


Device: cuda


## Paths

Adjust `MODEL_DIR` to point to your uploaded model dataset.

In [3]:
# Competition data
BASE_PATH = Path("/kaggle/input/competitions/hms-harmful-brain-activity-classification")

# ============================================================
# ADJUST THIS: point to your uploaded model checkpoint dataset
# Upload best_model_fold0.pt (and other folds) as a Kaggle dataset
# ============================================================
MODEL_DIR = Path("/kaggle/input/models/andrewpearlee/test/pytorch/default/1")  # <-- change this

# Working directory
PROCESSED_DIR = Path("/kaggle/temp/processed_eeg")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Find available checkpoints
ckpt_files = sorted(MODEL_DIR.glob("best_model_fold*.pt"))
print(f"Found {len(ckpt_files)} model checkpoints:")
for f in ckpt_files:
    print(f"  {f.name}")


Found 1 model checkpoints:
  best_model_fold0.pt


## Load Test Metadata

In [4]:
test_df = pd.read_csv(BASE_PATH / "test.csv")
test_df["eeg_path"] = test_df["eeg_id"].apply(
    lambda x: str(BASE_PATH / "test_eegs" / f"{x}.parquet")
)
print(f"Test samples: {len(test_df)}")
print(test_df.head())


Test samples: 1
   spectrogram_id      eeg_id  patient_id  \
0          853520  3911565283        6885   

                                            eeg_path  
0  /kaggle/input/competitions/hms-harmful-brain-a...  


## Bipolar Montage

In [5]:
BIPOLAR_MONTAGE = {
    "LL": [("Fp1", "F7"), ("F7", "T3"), ("T3", "T5"), ("T5", "O1")],
    "RL": [("Fp2", "F8"), ("F8", "T4"), ("T4", "T6"), ("T6", "O2")],
    "LP": [("Fp1", "F3"), ("F3", "C3"), ("C3", "P3"), ("P3", "O1")],
    "RP": [("Fp2", "F4"), ("F4", "C4"), ("C4", "P4"), ("P4", "O2")],
}

BIPOLAR_PAIRS = []
CHAIN_ORDER = ["LL", "RL", "LP", "RP"]
for chain_name in CHAIN_ORDER:
    BIPOLAR_PAIRS.extend(BIPOLAR_MONTAGE[chain_name])

print(f"Total bipolar channels: {len(BIPOLAR_PAIRS)}")


Total bipolar channels: 16


## Preprocess Test EEGs

In [6]:
def bandpass_filter(data, low, high, fs, order=4):
    nyq = fs / 2.0
    sos = butter(order, [low / nyq, high / nyq], btype='band', output='sos')
    filtered = np.zeros_like(data)
    for i in range(data.shape[0]):
        if np.std(data[i]) > 1e-6:
            try:
                filtered[i] = sosfiltfilt(sos, data[i]).astype(np.float32)
            except ValueError:
                filtered[i] = data[i]
    return filtered


def preprocess_eeg(row):
    eeg_id = row["eeg_id"]
    output_path = PROCESSED_DIR / f"{eeg_id}.npz"
    if output_path.exists():
        return

    eeg_df = pd.read_parquet(row["eeg_path"])
    offset = int(row.get("eeg_label_offset_seconds", 0))
    start = offset * CFG.eeg_sample_rate
    end = start + CFG.eeg_samples
    window = eeg_df.iloc[start:end]

    if len(window) < CFG.eeg_samples:
        pad = pd.DataFrame(
            np.zeros((CFG.eeg_samples - len(window), len(window.columns))),
            columns=window.columns,
        )
        window = pd.concat([window, pad], ignore_index=True)

    cols = window.columns.tolist()
    bipolar = []
    for (a, b) in BIPOLAR_PAIRS:
        if a in cols and b in cols:
            sig = window[a].values - window[b].values
        else:
            sig = np.zeros(CFG.eeg_samples, dtype=np.float32)
        bipolar.append(sig)
    bipolar = np.stack(bipolar, axis=0).astype(np.float32)

    bipolar = np.nan_to_num(bipolar, nan=0.0)
    bipolar = np.clip(bipolar, -1024, 1024)

    bipolar = bandpass_filter(
        bipolar, CFG.bandpass_low, CFG.bandpass_high,
        CFG.eeg_sample_rate, CFG.bandpass_order
    )

    chan_mean = bipolar.mean(axis=1).astype(np.float32)
    chan_std = bipolar.std(axis=1).astype(np.float32)
    stats = np.concatenate([chan_mean, chan_std])

    bipolar = (bipolar - chan_mean[:, None]) / (chan_std[:, None] + 1e-6)
    np.savez_compressed(str(output_path), eeg=bipolar, stats=stats)


print(f"Preprocessing {len(test_df)} test EEGs...")
for i in tqdm(range(len(test_df))):
    preprocess_eeg(test_df.iloc[i])
print("Done.")


Preprocessing 1 test EEGs...


  0%|          | 0/1 [00:00<?, ?it/s]

Done.


## Dataset

In [7]:
class HMSDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
        self.npz_paths = [
            str(PROCESSED_DIR / f"{eid}.npz")
            for eid in self.df["eeg_id"].values
        ]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        data = np.load(self.npz_paths[idx])
        eeg = torch.tensor(data["eeg"], dtype=torch.float32)
        stats = torch.tensor(data["stats"], dtype=torch.float32)
        return eeg, stats

test_dataset = HMSDataset(test_df)
test_loader = DataLoader(
    test_dataset,
    batch_size=CFG.batch_size,
    shuffle=False,
    num_workers=CFG.num_workers,
    pin_memory=True,
    drop_last=False,
)
print(f"Test batches: {len(test_loader)}")


Test batches: 1


## Model Architecture

In [8]:
class SpectrogramModel(nn.Module):
    def __init__(self, cfg=CFG):
        super().__init__()
        self.cfg = cfg

        freq_res = cfg.eeg_sample_rate / cfg.n_fft
        self.max_bin = int(cfg.freq_crop_hz / freq_res) + 1

        self.zoom_samples = cfg.zoom_duration * cfg.eeg_sample_rate
        self.zoom_start = (cfg.eeg_samples - self.zoom_samples) // 2
        self.zoom_end = self.zoom_start + self.zoom_samples

        self.stft_wide = STFT(
            n_fft=cfg.n_fft, hop_length=cfg.hop_length,
            sr=cfg.eeg_sample_rate, trainable=cfg.trainable_stft,
            output_format="Magnitude",
        )
        self.stft_zoom = STFT(
            n_fft=cfg.n_fft, hop_length=cfg.zoom_hop_length,
            sr=cfg.eeg_sample_rate, trainable=cfg.trainable_stft,
            output_format="Magnitude",
        )
        self.log_eps = 1e-6

        self.backbone = timm.create_model(
            cfg.backbone, pretrained=False, in_chans=3, features_only=True,
        )
        img_h = 16 * self.max_bin
        with torch.no_grad():
            dummy = torch.randn(1, 3, img_h, 64)
            backbone_out = self.backbone(dummy)
        backbone_channels = backbone_out[-1].shape[1]

        self.gru = nn.GRU(
            input_size=backbone_channels, hidden_size=cfg.gru_hidden,
            num_layers=cfg.gru_layers, batch_first=True, bidirectional=True,
        )

        gru_out_dim = cfg.gru_hidden * 2
        self.time_attn = nn.Linear(gru_out_dim, 1)

        self.stats_mlp = nn.Sequential(
            nn.Linear(32, 64), nn.ReLU(),
            nn.Dropout(cfg.dropout), nn.Linear(64, 32),
        )

        self.head = nn.Sequential(
            nn.Dropout(cfg.dropout),
            nn.Linear(gru_out_dim + 32, 128), nn.ReLU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(128, cfg.num_classes),
        )

    def _run_stft(self, eeg, stft_layer):
        batch, channels, time = eeg.shape
        x = eeg.reshape(batch * channels, time)
        x = stft_layer(x)
        x = x[:, :self.max_bin, :]
        freq_bins, time_frames = x.shape[1], x.shape[2]
        return x.reshape(batch, channels, freq_bins, time_frames)

    def _stack_channels(self, spec):
        batch, channels, freq, time = spec.shape
        return spec.reshape(batch, channels * freq, time)

    def _log_norm(self, spec):
        x = torch.log(spec.clamp(min=self.log_eps))
        mean = x.mean(dim=(1, 2), keepdim=True)
        std = x.std(dim=(1, 2), keepdim=True) + 1e-6
        return (x - mean) / std

    def make_multiscale_image(self, eeg):
        spec_wide = self._run_stft(eeg, self.stft_wide)
        wide = self._log_norm(self._stack_channels(spec_wide))

        eeg_zoom = eeg[:, :, self.zoom_start:self.zoom_end]
        spec_zoom = self._run_stft(eeg_zoom, self.stft_zoom)
        zoom = self._log_norm(self._stack_channels(spec_zoom))

        T_wide = wide.shape[2]
        zoom = F.interpolate(
            zoom.unsqueeze(1), size=(wide.shape[1], T_wide),
            mode='bilinear', align_corners=False,
        ).squeeze(1)

        mean_spec = (wide + zoom) / 2.0
        return torch.stack([wide, zoom, mean_spec], dim=1)

    def forward(self, eeg, stats):
        img = self.make_multiscale_image(eeg)
        features = self.backbone(img)
        fmap = features[-1]

        x = fmap.mean(dim=2).permute(0, 2, 1)
        x = x.float()
        x, _ = self.gru(x)

        w = torch.softmax(self.time_attn(x), dim=1)
        x = (x * w).sum(dim=1)

        s = self.stats_mlp(stats.float())
        combined = torch.cat([x, s], dim=1)
        logits = self.head(combined)
        return F.softmax(logits, dim=1)

print("Model defined.")


Model defined.


## Inference — Ensemble All Folds

In [9]:
@torch.no_grad()
def predict(model, loader, device):
    model.eval()
    all_preds = []
    for eeg, stats in tqdm(loader, desc="Predicting"):
        eeg = eeg.to(device, non_blocking=True)
        stats = stats.to(device, non_blocking=True)
        with autocast(enabled=CFG.use_amp):
            preds = model(eeg, stats)
        all_preds.append(preds.cpu().numpy())
    return np.concatenate(all_preds, axis=0)


all_fold_preds = []

for ckpt_path in ckpt_files:
    fold_name = ckpt_path.stem
    print(f"\n{fold_name}:")
    gc.collect()
    torch.cuda.empty_cache()

    model = SpectrogramModel(CFG).to(CFG.device)
    ckpt = torch.load(str(ckpt_path), map_location=CFG.device)
    model.load_state_dict(ckpt["model_state_dict"])
    print(f"  Loaded (val_loss={ckpt.get('val_loss', '?'):.4f})")

    fold_preds = predict(model, test_loader, CFG.device)
    all_fold_preds.append(fold_preds)
    print(f"  Predictions: {fold_preds.shape}")

    del model
    gc.collect()
    torch.cuda.empty_cache()

preds = np.mean(all_fold_preds, axis=0)
print(f"\nEnsemble of {len(all_fold_preds)} model(s)")
print(f"Predictions shape: {preds.shape}")
print(f"Prob sum check: {preds[0].sum():.4f}")



best_model_fold0:
STFT kernels created, time used = 0.0587 seconds
STFT kernels created, time used = 0.0021 seconds
  Loaded (val_loss=1.0227)


Predicting:   0%|          | 0/1 [00:00<?, ?it/s]

/tmp/ipykernel_24/1078336110.py:8: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=CFG.use_amp):


  Predictions: (1, 6)

Ensemble of 1 model(s)
Predictions shape: (1, 6)
Prob sum check: 1.0000


Exception in thread QueueFeederThread:
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 259, in _feed
    reader_close()
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 178, in close
    self._close()
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 377, in _close
    _close(self._handle)
OSError: [Errno 9] Bad file descriptor

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 291, in _feed
    queue_sem.release()
ValueError: semaphore or lock released too many times


## Create Submission

In [10]:
target_cols = [x.lower() + "_vote" for x in CFG.class_names]
pred_df = test_df[["eeg_id"]].copy()
pred_df[target_cols] = preds.tolist()

sub_df = pd.read_csv(BASE_PATH / "sample_submission.csv")
sub_df = sub_df[["eeg_id"]].copy()
sub_df = sub_df.merge(pred_df, on="eeg_id", how="left")

sub_df.to_csv("submission.csv", index=False)
print(f"submission.csv saved ({len(sub_df)} rows)")
print(sub_df.head())


submission.csv saved (1 rows)
       eeg_id  seizure_vote  lpd_vote  gpd_vote  lrda_vote  grda_vote  \
0  3911565283      0.012854  0.045393  0.014779    0.14204   0.222694   

   other_vote  
0     0.56224  
